In [ ]:
import numpy as np
import pandas as pd

!pip install facenet-pytorch


In [ ]:
from tqdm import tqdm
import os
from facenet_pytorch import MTCNN
from PIL import Image
from torchvision import transforms
import torch

In [ ]:
from torchvision import transforms

transform_train = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5]),
])

transform_dev = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5]),
])

transform_test = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5]),
])


In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
! kaggle datasets download -d jessicali9530/lfw-dataset

In [ ]:
!unzip lfw-dataset.zip -d ./data

In [ ]:
INPUT_DIR = "./data/lfw-deepfunneled/lfw-deepfunneled"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

mtcnn = MTCNN(image_size=160, margin=20, device=device)

def extract_faces(input_dir, output_dir):
    """
    Detects and crops faces from images using MTCNN.
    """
    os.makedirs(output_dir, exist_ok=True)
    for person in tqdm(os.listdir(input_dir)):
        person_path = os.path.join(input_dir, person)
        if not os.path.isdir(person_path): continue
        save_path = os.path.join(output_dir, person)
        os.makedirs(save_path, exist_ok=True)
        for img_name in os.listdir(person_path):
            img_path = os.path.join(person_path, img_name)
            try:
                img = Image.open(img_path).convert("RGB")
                face = mtcnn(img)
                if face is not None:
                    face_pil = transforms.ToPILImage()(face.cpu().detach())
                    safe_name = os.path.splitext(img_name)[0] + ".jpg"
                    face_pil.save(os.path.join(save_path, safe_name))
            except Exception as e:
                print(f"Error on {img_path}: {e}")


In [ ]:
import shutil
import random

def split_dataset(root_dir, output_dir, seed = 42):
    random.seed(seed)
    persons = [d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))]
    random.shuffle(persons)

    train_num = int(len(persons) * 0.8)
    dev_num = train_num + int(len(persons) * 0.1)

    splits = {
        "train": persons[:train_num],
        "dev": persons[train_num:dev_num],
        "test": persons[dev_num:]
    }

    for split_name, person_list in splits.items() :
        split_path = os.path.join(output_dir, split_name)
        os.makedirs(split_path, exist_ok = True)
        for person in person_list :
            src = os.path.join(root_dir, person)
            dst = os.path.join(split_path, person)
            shutil.copytree(src, dst, dirs_exist_ok=True)


In [ ]:
from torch.utils.data import Dataset
from PIL import Image

class TripletFaceDataset(Dataset):
    def __init__(self, root_dir, transform = None) :
        self.root_dir = root_dir
        self.transform = transform

        #person -> images
        self.person_to_images = {}
        for person in os.listdir(root_dir) :
            person_path = os.path.join(root_dir, person)
            if os.path.isdir(person_path) :
                images = [os.path.join(person_path, f) for f in os.listdir(person_path)]
                if len(images) >= 2:
                    self.person_to_images[person] = images

        #persons with atleast 2 image (becuase we have anchor and positive)
        self.persons = list(self.person_to_images.keys())

        #images using for negative images
        self.all_images = [(person, img) for person, imgs in self.person_to_images.items() for img in imgs]


    def __len__(self) :
        return sum(len(imgs) for imgs in self.person_to_images.values())

    def __getitem__(self, idx):
        #selecting anchor person and image randomly
        person = self.persons[idx % len(self.persons)]
        anchor_img = random.choice(self.person_to_images[person])

        #selecting positive image (it should not be the anchor image)
        positive_img = anchor_img
        while positive_img == anchor_img:
            positive_img = random.choice(self.person_to_images[person])

        #selecting neagtive person (it should not be the anchor person)
        negative_person = person
        while negative_person == person:
            negative_person = random.choice(self.persons)
        negative_img = random.choice(self.person_to_images[negative_person])

        #loading images and transform them
        anchor = Image.open(anchor_img).convert("RGB")
        positive = Image.open(positive_img).convert("RGB")
        negative = Image.open(negative_img).convert("RGB")

        if self.transform:
            anchor = self.transform(anchor)
            positive = self.transform(positive)
            negative = self.transform(negative)

        return anchor, positive, negative


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SiameseNetwork(nn.Module) :
    def __init__(self, embedding_size=128) :
        super(SiameseNetwork, self).__init__()

        self.embedding_size = embedding_size

        self.convnet = nn.Sequential (
            nn.Conv2d(in_channels = 3, out_channels = 32, kernel_size = 7, stride = 1, padding = 3),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size = 5, stride = 1, padding = 2),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(256, 512, kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.fc = nn.Linear(512, embedding_size)

    def forward(self, x) :
        x = self.convnet(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        x = F.normalize(x, p = 2, dim = 1)
        return x

In [ ]:
from torch.utils.data import DataLoader


INPUT_DIR = "./data/lfw-deepfunneled/lfw-deepfunneled"
extract_faces(INPUT_DIR, "./aligned_faces")

model = SiameseNetwork(embedding_size = 128).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-3)
triplet_loss = nn.TripletMarginLoss(margin = 1.0, p = 2)

In [ ]:
split_dataset("./aligned_faces", "./split_faces")

base_dir = "./split_faces"
train_dataset = TripletFaceDataset(root_dir=os.path.join(base_dir, "train"), transform=transform_train)
dev_dataset = TripletFaceDataset(root_dir=os.path.join(base_dir, "dev"), transform=transform_dev)
test_dataset = TripletFaceDataset(root_dir=os.path.join(base_dir, "test"), transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
model.train()

train_loss = 0.0

for epoch in range(10) :
    for batch_idx, (anchor, positive, negative) in enumerate(train_loader) :
        anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)
        anchor_out = model(anchor)
        positive_out = model(positive)
        negative_out = model(negative)
        loss = triplet_loss(anchor_out, positive_out, negative_out)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

        if batch_idx % 10 == 0:
            print(f"Epoch {epoch}, Batch {batch_idx}, Loss: {loss.item():.4f}")


In [ ]:
torch.save(model.state_dict(), "face_recognition_model.pth")

model = SiameseNetwork(embedding_size=128)
model.load_state_dict(torch.load("face_recognition_model.pth"))
model.eval()

correct_total = 0
total_samples = 0

with torch.no_grad():
    for anchor, positive, negative in dev_loader:
        anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)
        anchor_out = model(anchor)
        positive_out = model(positive)
        negative_out = model(negative)

        loss = triplet_loss(anchor_out, positive_out, negative_out)

        d_pos = F.pairwise_distance(anchor_out, positive_out)
        d_neg = F.pairwise_distance(anchor_out, negative_out)
        correct = (d_pos < d_neg).sum().item()
        total = anchor_out.size(0)

        correct_total += correct
        total_samples += total

accuracy = correct_total / total_samples
error_percent = 100 * (1 - accuracy)

print(f"Dev Accuracy: {accuracy:.4f}, Dev Error: {error_percent:.2f}%")